In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import re
from tabulate import tabulate

In [2]:
params_data = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,34276;"
    "DATABASE=Ventas_Shopify;"
    "UID=sa;"
    "PWD=L0cc1T4n3*$*;"
)

# Crear el motor de conexión
engine_data = create_engine(f"mssql+pyodbc:///?odbc_connect={params_data}")

# Configurar cadena de conexión
params_com = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=comerssiamirror.eastus2.cloudapp.azure.com,38693;"
    "DATABASE=PROVENZAL;"
    "UID=provenzal;"
    "PWD=PE@4:BRy/<1VWkp;"
)

# Crear el motor
engine_com = create_engine(f"mssql+pyodbc:///?odbc_connect={params_com}")

query_data = """
SELECT DISTINCT 
    v.Id_Cliente AS Cliente,
    v.Fecha AS Fecha_Venta,
    v.Canal AS Canal_Recompra,
	h.Categoria,
	h.Venta_Neta AS Venta
FROM Ventas_ShopifyMoviNova v 
INNER JOIN Venta_Homologada h ON v.Pedido = h.Pedido
WHERE v.fecha >= '2025-05-01'
    --AND Fecha <= '2026-02-28'
    AND Canal != 'MARKETPLACE'
    AND h.Venta_Neta > 0
"""

# Ejecutar y cargar en DataFrame
ventas_shopify = pd.read_sql(query_data, engine_data)

query_com = """
SELECT DISTINCT 
    v.Cliente AS Cliente,
    v.Fecha AS Fecha_Venta,
    v.Canal AS Canal_Recompra,
    v.Categoria,
    v.Venta_Neta AS Venta
FROM Ventas_MoviNova v 
WHERE v.fecha >= '2025-05-01'
    --AND Fecha <= '2026-02-28'
    AND Canal != 'C&C' 
    AND Canal != 'RAPPI'
    AND v.Venta_Neta > 0
"""

# Ejecutar y cargar en DataFrame
ventas_comerssia = pd.read_sql(query_com, engine_com)

In [3]:
mapa_categorias = {
    '00002': 'ARTICULOS DE ASEO',
    '00003': 'CUIDADO CAPILAR',
    '00004': 'CUIDADO CORPORAL',
    '00005': 'CUIDADO FACIAL',
    '00006': 'CUIDADO DE MANOS',
    '00007': 'HOGAR',
    '00009': 'OTROS',
    '00010': 'FRAGANCIAS',
    '00016': 'KIT MULTICATEGORÍA'
}

ventas_comerssia['Categoria'] = ventas_comerssia['Categoria'].map(mapa_categorias)

df_ventas = pd.concat([ventas_comerssia, ventas_shopify], ignore_index=True)


In [4]:
# Normalizar el nombre del canal
df_ventas['Canal_Recompra'] = df_ventas['Canal_Recompra'].replace({
    'Chat Center': 'Social Selling',
    'CHATCENTER WEB': 'Social Selling'
})


def limpiar_codigo(codigo):
    codigo = re.sub(r'^C\s*', '', codigo, flags=re.IGNORECASE) # elimina 'C' + cualquier número de espacios al inicio
    codigo = str(codigo).strip()              # quita espacios al inicio y al final
    codigo = codigo.replace(' ', '')          # elimina cualquier espacio restante en el medio
    return codigo

#Conviertir a String y aplicar funcion para lipiar codigo
df_ventas['Cliente'] = df_ventas['Cliente'].apply(limpiar_codigo)

df_ventas['Cliente'] = df_ventas['Cliente'].astype(str).str.strip().str.upper()
df_ventas['Cliente'] = df_ventas['Cliente'].apply(lambda x: x if x.startswith('C') else f'C{x}')
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

df_ventas

,Cliente,Fecha_Venta,Canal_Recompra,Categoria,Venta
0,C1125248087,2025-05-12,Retail,ARTICULOS DE ASEO,57983.19
1,C00112083407,2025-11-09,Retail,ARTICULOS DE ASEO,109243.70
2,C00112704010,2025-09-18,Retail,FRAGANCIAS,289075.63
3,C00113169361,2026-03-06,Retail,CUIDADO CORPORAL,176470.59
4,C00116289554,2026-01-23,Retail,FRAGANCIAS,136134.45
...,...,...,...,...,...
270458,CSCOTIABANKCOLPATRIA,2025-12-01,Shopify,None,214276.50
270459,CTHEKEY,2025-09-04,Shopify,FRAGANCIAS,297466.20
270460,CXDD863757,2025-10-18,Shopify,ARTICULOS DE ASEO,98315.10
270461,CYB0138294,2025-11-27,Social Selling,None,21007.50


In [5]:
# Cargar el archivo Excel
ruta_excel = 'PLAN 120.xlsx'
plan120_excel = pd.read_excel(ruta_excel,sheet_name=None)

In [6]:
# ▶️ Inicializar diccionario de resultados
resultados = {}

# ▶️ Recorrer cada hoja (mes) del Plan 120
for mes, df_nuevos in plan120_excel.items():
    # Normalizar cliente y fecha ingreso
    df_nuevos['Cliente'] = df_nuevos['Cliente'].astype(str).str.strip().str.upper()
    df_nuevos['Cliente'] = df_nuevos['Cliente'].apply(lambda x: x if x.startswith('C') else f'C{x}')
    df_nuevos['Fecha_Ingreso'] = pd.to_datetime(df_nuevos['Fecha_Ingreso']).dt.date

    # ▶️ Unir con df_ventas para quedarnos solo con ventas posteriores al ingreso
    df_merge = df_ventas.merge(df_nuevos, on='Cliente', how='inner')

    # Filtrar ventas después de la fecha de ingreso
    df_recompras = df_merge[df_merge['Fecha_Venta'] > df_merge['Fecha_Ingreso']].copy()

    # Paso 3: Obtener primera fecha de recompra por cliente
    primeras_fechas = df_recompras.groupby('Cliente')['Fecha_Venta'].min().reset_index()
    primeras_fechas.rename(columns={'Fecha_Venta': 'FechaRecompra'}, inplace=True)

    # Paso 4: Unir con df_recompras para quedarnos con solo compras del mismo día
    df_primera_recompra = df_recompras.merge(primeras_fechas, on='Cliente')
    df_primera_recompra = df_primera_recompra[
        df_primera_recompra['Fecha_Venta'] == df_primera_recompra['FechaRecompra']
    ].copy()

    # Agregar fecha de ingreso
    df_primera_recompra = df_nuevos[['Cliente']].merge(
        df_primera_recompra, on='Cliente', how='inner'
    )

    # Agrupar por cliente + canal + categoría
    df_final = df_primera_recompra.groupby(
        ['Cliente', 'Fecha_Ingreso', 'Canal_Ingreso','FechaRecompra', 'Canal_Recompra', 'Categoria'], as_index=False
    )['Venta'].sum()

    # Marcar cambio de canal
    df_final['CambioCanal'] = df_final.apply(
        lambda row: 'NO CAMBIO' if row['Canal_Ingreso'] == row['Canal_Recompra'] else 'CAMBIO',
        axis=1
    )

     # Guardar resultado final
    resultados[mes] = df_final

     # 📊 Reporte de resumen por mes
    total_recompradores = df_final['Cliente'].nunique()
    cambios = df_final[df_final['CambioCanal'] == 'CAMBIO']['Cliente'].nunique()
    sin_cambio = total_recompradores - cambios

    print(f"\n📅 MES: {mes.upper()}")
    print(f"➡️ Clientes que recompraron: {total_recompradores}")
    print(f"🔁 Clientes que CAMBIARON de canal: {cambios}")
    print(f"✅ Clientes que NO cambiaron de canal: {sin_cambio}")

    # Detalle por cambio de canal
    detalle_cambios = df_final[df_final['CambioCanal'] == 'CAMBIO'].groupby(
        ['Canal_Ingreso', 'Canal_Recompra']
    )['Cliente'].nunique().reset_index(name='Clientes')

    if not detalle_cambios.empty:
        print("\n🔎 Detalle de cambios de canal:")
        print(tabulate(detalle_cambios, headers='keys', tablefmt='tsv', showindex=False))
    else:
        print("\n✔️ No hubo cambios de canal.")

    # ▶️ Agregar columna con nombre del mes de la recompra
    df_final['MesRecompra'] = pd.to_datetime(df_final['FechaRecompra']).dt.strftime('%B').str.upper()

    # ▶️ Calcular resumen de clientes únicos por mes de recompra
    recuento_clientes = df_final.groupby('MesRecompra')['Cliente'].nunique().reset_index()
    recuento_clientes.columns = ['MesRecompra', 'ClientesUnicos']

    print(f"\n📆 Recompras para Plan 120 ({mes.upper()}):")
    print(tabulate(recuento_clientes, headers='keys', tablefmt='tsv', showindex=False))


📅 MES: NOVIEMBRE
➡️ Clientes que recompraron: 403
🔁 Clientes que CAMBIARON de canal: 64
✅ Clientes que NO cambiaron de canal: 339

🔎 Detalle de cambios de canal:
Canal_Ingreso  	Canal_Recompra  	  Clientes
Retail         	Ferias          	         1
Retail         	Shopify         	        14
Retail         	Social Selling  	         6
Retail         	Whatsapp        	         1
Shopify        	Ferias          	         1
Shopify        	Retail          	        34
Shopify        	Social Selling  	         2
Social Selling 	Retail          	         3
Whatsapp       	Retail          	         2

📆 Recompras para Plan 120 (NOVIEMBRE):
MesRecompra  	  ClientesUnicos
DECEMBER     	             191
FEBRUARY     	              27
JANUARY      	              65
MARCH        	              17
NOVEMBER     	             103

📅 MES: DICIEMBRE
➡️ Clientes que recompraron: 565
🔁 Clientes que CAMBIARON de canal: 61
✅ Clientes que NO cambiaron de canal: 504

🔎 Detalle de cambios de canal:
Canal_In

In [7]:
# ▶️ Exportar a Excel final
with pd.ExcelWriter("plan_120_recompras_detallado.xlsx") as writer:
    for mes, df in resultados.items():
        df.to_excel(writer, sheet_name=mes[:31], index=False)
